In [2]:
import os

from dotenv import load_dotenv
from langchain_sarvam import ChatSarvam

In [3]:

load_dotenv()

if not os.getenv("SARVAM_API_KEY"):
    raise RuntimeError("Set SARVAM_API_KEY in your environment or .env file before running.")

llm = ChatSarvam(model="sarvam-105b")

In [4]:
response = llm.invoke("Write a 4 line poem about the beauty of nature.")

print(response.content)

The light spills through the forest green,
A gentle, soft and cooling breeze,
A quiet, vibrant, living scene,
That puts the hurried mind at ease.


In [6]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path="codebasics_faqs.csv",
    source_column="prompt",
    encoding="latin-1"
)

data = loader.load()

print(data[0])

/var/folders/jp/xvts_ymx1lb_nf8vz1xmd8l00000gn/T/ipykernel_36577/677344369.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


page_content='prompt: I have never done programming in my life. Can I take this bootcamp?
response: Yes, this is the perfect bootcamp for anyone who has never done coding and wants to build a career in the IT/Data Analytics industry or just wants to perform better in your current job or business using data.' metadata={'source': 'I have never done programming in my life. Can I take this bootcamp?', 'row': 0}


In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

e = embeddings.embed_query("What is your refund policy?")

print(e[:5])

/var/folders/jp/xvts_ymx1lb_nf8vz1xmd8l00000gn/T/ipykernel_36577/946932511.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4657.07it/s]


[-0.07025407999753952, 0.04848405346274376, 0.04861599951982498, 0.018366435542702675, 0.0577784888446331]


In [8]:
len(e)

384

In [9]:
from langchain_community.vectorstores import FAISS
# Create a FAISS instance for vector database from 'data'
# vectordb = FAISS.from_documents(documents=data,
#                                  embedding=instructor_embeddings)

# # Create a retriever for querying the vector database
# retriever = vectordb.as_retriever(score_threshold = 0.7)

from langchain_community.vectorstores import FAISS

vectordb = FAISS.from_documents(
    documents=data,
    embedding=embeddings
)

# Create a retriever for querying the vector database
retriever = vectordb.as_retriever(score_threshold = 0.7)

In [10]:
rdocs = retriever.invoke("how about job placement support?")

for doc in rdocs:
    print(doc.page_content)
    print("----------------")

prompt: Do you provide any job assistance?
response: Yes, We help you with resume and interview preparation along with that we help you in building online credibility, and based on requirements we refer candidates to potential recruiters.
----------------
prompt: Will this course guarantee me a job?
response: We created a much lighter version of this course on YouTube available for free (click this link) and many people gave us feedback that they were able to fetch jobs (see testimonials). Now this paid course is at least 5x better than the YouTube course which gives us ample confidence that you will be able to get a job. However, we want to be honest and do not want to make any impractical promises! Our guarantee is to prepare you for the job market by teaching the most relevant skills, knowledge & timeless principles good enough to fetch the job.
----------------
prompt: How do become good and comfortable with SQL?
response: To master SQL, we need to practice it every day and keep in

In [11]:
from langchain_core.prompts import PromptTemplate

prompt_template = """Given the following context and a question, generate an answer based on this context only.
In the answer try to provide as much text as possible from "response" section in the source document context without making much changes.
If the answer is not found in the context, kindly state "I don't know." Don't try to make up an answer.

CONTEXT: {context}

QUESTION: {question}"""


PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)
chain_type_kwargs = {"prompt": PROMPT}


from langchain_classic.chains import RetrievalQA

chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={
        "prompt": PROMPT
    }
)


In [12]:
chain('Do you provide job assistance and also do you provide job gurantee?')

/var/folders/jp/xvts_ymx1lb_nf8vz1xmd8l00000gn/T/ipykernel_36577/2066545439.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  chain('Do you provide job assistance and also do you provide job gurantee?')


{'query': 'Do you provide job assistance and also do you provide job gurantee?',
 'result': 'Yes, We help you with resume and interview preparation along with that we help you in building online credibility, and based on requirements we refer candidates to potential recruiters. However, we want to be honest and do not want to make any impractical promises! Our guarantee is to prepare you for the job market by teaching the most relevant skills, knowledge & timeless principles good enough to fetch the job.'}

In [13]:
chain("do you have javascript course?")

{'query': 'do you have javascript course?', 'result': "I don't know."}